In [31]:
import pandas as pd

In [32]:
df = pd.read_csv('../data/processed/cleaned_supply_chain.csv')

In [33]:
#check if the datatype still preserve for datetime
df.dtypes[['order date (DateOrders)', 'shipping date (DateOrders)']]

order date (DateOrders)       object
shipping date (DateOrders)    object
dtype: object

In [34]:
df.columns.tolist()

['Type',
 'Days for shipping (real)',
 'Days for shipment (scheduled)',
 'Benefit per order',
 'Sales per customer',
 'Delivery Status',
 'Late_delivery_risk',
 'Category Name',
 'Customer City',
 'Customer Country',
 'Customer Id',
 'Customer Segment',
 'Customer State',
 'Customer Zipcode',
 'Department Name',
 'Latitude',
 'Longitude',
 'Market',
 'Order City',
 'Order Country',
 'Order Customer Id',
 'order date (DateOrders)',
 'Order Id',
 'Order Item Discount',
 'Order Item Discount Rate',
 'Order Item Id',
 'Order Item Product Price',
 'Order Item Profit Ratio',
 'Order Item Quantity',
 'Sales',
 'Order Item Total',
 'Order Profit Per Order',
 'Order Region',
 'Order State',
 'Order Status',
 'Product Card Id',
 'Product Name',
 'Product Price',
 'Product Status',
 'shipping date (DateOrders)',
 'Shipping Mode']

In [35]:
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])
df['shipping date (DateOrders)'] = pd.to_datetime(df['shipping date (DateOrders)'])                                        

In [36]:
df.dtypes[['order date (DateOrders)', 'shipping date (DateOrders)']]

order date (DateOrders)       datetime64[ns]
shipping date (DateOrders)    datetime64[ns]
dtype: object

In [37]:
df[['order date (DateOrders)', 'shipping date (DateOrders)']].head()

,order date (DateOrders),shipping date (DateOrders)
0,2018-01-31 22:56:00,2018-02-03 22:56:00
1,2018-01-13 12:27:00,2018-01-18 12:27:00
2,2018-01-13 12:06:00,2018-01-17 12:06:00
3,2018-01-13 11:45:00,2018-01-16 11:45:00
4,2018-01-13 11:24:00,2018-01-15 11:24:00


In [38]:
df['Delivery Status'].value_counts()

Delivery Status
Late delivery        98976
Advance shipping     41592
Shipping on time     32194
Shipping canceled     7754
Name: count, dtype: int64

In [39]:
df['Delivery Status'].value_counts(normalize = True)

Delivery Status
Late delivery        0.548295
Advance shipping     0.230406
Shipping on time     0.178344
Shipping canceled    0.042955
Name: proportion, dtype: float64

In [40]:
df.groupby('Shipping Mode')['Late_delivery_risk'].mean().sort_values(ascending = False)

Shipping Mode
First Class       0.953223
Second Class      0.766328
Same Day          0.457430
Standard Class    0.380724
Name: Late_delivery_risk, dtype: float64

In [41]:
df.groupby('Shipping Mode')[['Days for shipment (scheduled)', 'Days for shipping (real)']].mean()

,Days for shipment (scheduled),Days for shipping (real)
Shipping Mode,,
First Class,1.0,2.000000
Same Day,0.0,0.478279
Second Class,2.0,3.990828
Standard Class,4.0,3.995907


In [42]:
df.groupby(['Market', 'Shipping Mode'])[['Days for shipment (scheduled)', 'Days for shipping (real)']].mean()

Days for shipment (scheduled)  \
Market       Shipping Mode                                   
Africa       First Class                               1.0   
             Same Day                                  0.0   
             Second Class                              2.0   
             Standard Class                            4.0   
Europe       First Class                               1.0   
             Same Day                                  0.0   
             Second Class                              2.0   
             Standard Class                            4.0   
LATAM        First Class                               1.0   
             Same Day                                  0.0   
             Second Class                              2.0   
             Standard Class                            4.0   
Pacific Asia First Class                               1.0   
             Same Day                                  0.0   
             Second Class                              2.0   
             Standard Class                            4.0   
USCA         First Class                               1.0   
             Same Day                                  0.0   
             Second Class                              2.0   
             Standard Class                            4.0   

                             Days for shipping (real)  
Market       Shipping Mode                             
Africa       First Class                     2.000000  
             Same Day                        0.474551  
             Second Class                    3.982440  
             Standard Class                  4.024096  
Europe       First Class                     2.000000  
             Same Day                        0.486046  
             Second Class                    4.003448  
             Standard Class                  3.989811  
LATAM        First Class                     2.000000  
             Same Day                        0.510943  
             Second Class                    3.981890  
             Standard Class                  3.994955  
Pacific Asia First Class                     2.000000  
             Same Day                        0.448338  
             Second Class                    3.998404  
             Standard Class                  3.996421  
USCA         First Class                     2.000000  
             Same Day                        0.451185  
             Second Class                    3.975318  
             Standard Class                  3.995869

### Finding
dataco's first class and second class shipping promises are structurally miscalibrated as both consistently take ~2x their promised delivery window, uniformly across all 5 markets, indicating the issue is a company-wide policy/SLA-setting problem rather than regional filfulllment variability. Standard Calss's promis is realistic and consistently met. Recommendation: either reset the promised delivery windows for First/Second Class to match actual fulfillment capability(2 days or 4 days), or investigate why fulfillment can't meet the current 1-day/2-day target. Starting from there would be good enough, since it affects every region identically, would have the broadest impact for the least investigative cost.

In [43]:
df.groupby('Late_delivery_risk')['Order Profit Per Order'].agg(['mean','count'])

,mean,count
Late_delivery_risk,,
0,22.403488,81540
1,21.621255,98976


In [44]:
df.groupby('Delivery Status')['Order Profit Per Order'].agg(['mean','count'])

,mean,count
Delivery Status,,
Advance shipping,22.485701,41592
Late delivery,21.621255,98976
Shipping canceled,20.696717,7754
Shipping on time,22.708355,32194


In [45]:
#look only at the rows where the shipment was cenceled and compare them to the description of Order Status
df[df['Delivery Status'] == 'Shipping canceled']['Order Status'].value_counts()

Order Status
SUSPECTED_FRAUD    4062
CANCELED           3692
Name: count, dtype: int64

In [46]:
df_fraud = df[df['Order Status'] == 'SUSPECTED_FRAUD']
df = df[df['Order Status'] != 'SUSPECTED_FRAUD']
df.shape

(176454, 41)

In [47]:
df.groupby('Shipping Mode')[['Days for shipment (scheduled)', 'Days for shipping (real)']].mean()

,Days for shipment (scheduled),Days for shipping (real)
Shipping Mode,,
First Class,1.0,2.000000
Same Day,0.0,0.478567
Second Class,2.0,3.991209
Standard Class,4.0,3.994760


### EDA Summary
overall late-delivery rate: 54.8% (98,977 of 180,516 orders, post-cleaning)
late delivery is heavily concentrated in First Class (95.3 % late) and Second Class (76.6% Late); Standard class (38.1%) and Same Day (45.7%) perform far better
Roo-casue: First class and second class are structurally over-promised - both consistently take about 2x their promised delivery window, iniformly across all 5 markets with near-zero variance. This points to a company-wide SLA-setting issue, not regional fulfillment problems
Standard Class, despite being the 'Slowest' promiesed tier, is the most reliable as its promise is realistic and consistently met
Profit per order is nearly identical between late and on-time orders (21.62 vs 22.40, a 3.5% gap) - the cost of lateness likely isn't in per-transaction margin, but in unmeasured downstream effects like retention (out of scope for this dataset)
4,062 rows flagges SUSPECTED_FRAUD were identified and excluded from delivery analysis as a separate business process

In [48]:
df.to_csv('../data/processed/cleaned_supply_chain_no_fraud.csv', index = False)

In [49]:
df['Late_delivery_risk'].mean()

np.float64(0.5609167261722602)

In [50]:
df.groupby(df['order date (DateOrders)'].dt.year)['order date (DateOrders)'].count()

order date (DateOrders)
2015    61265
2016    61174
2017    51933
2018     2082
Name: order date (DateOrders), dtype: int64

In [58]:
df[df['order date (DateOrders)'].dt.year == 2017]['order date (DateOrders)'].dt.month.value_counts().sort_index()

order date (DateOrders)
1     5108
2     4776
3     5253
4     5091
5     5160
6     4818
7     5183
8     5187
9     5070
10    2204
11    2006
12    2077
Name: count, dtype: int64

In [55]:
df[df['order date (DateOrders)'].dt.year == 2017].groupby(df['order date (DateOrders)'].dt.month)['order date (DateOrders)'].dt.day.nunique()

AttributeError: 'SeriesGroupBy' object has no attribute 'dt'

In [56]:
df[df['order date (DateOrders)'].dt.year == 2017].groupby(df['order date (DateOrders)'].dt.month)['order date (DateOrders)'].dt.day.nunique()

AttributeError: 'SeriesGroupBy' object has no attribute 'dt'

In [57]:
df_2017 = df[df['order date (DateOrders)'].dt.year == 2017].copy()
df_2017['order_day'] = df_2017['order date (DateOrders)'].dt.day
df_2017.groupby(df_2017['order date (DateOrders)'].dt.month)['order_day'].nunique()

order date (DateOrders)
1     31
2     28
3     31
4     30
5     31
6     30
7     31
8     31
9     30
10    31
11    30
12    31
Name: order_day, dtype: int64

Order volume drops sharply starting October 2017 (5,000+ a month to 2,000-2,200/month) and stays at that lower level through the end of the dataset. Verified this isn't a data-completeness artifact - every day in oct-dec 2017 to jan of 2018 has orders recorded. Root cause is unknown without access to DataCo's actual business records. This will need explicit handling in the demand forecasting notebook - likely training primarily on the post-drop period, or flaggig forecast uncertainty - rather than treating the full 2015 - 2018 range as one consistent pattern